# CEMAT-Stack V4 — Publication Run

Run all cells. This notebook executes the checksum-pinned leakage-safe 5×5 repeated nested-CV pipeline on the complete 463-patient prognostic cohort. A GPU is not required. Completed outer folds resume from Google Drive.

**After completion, the final cell displays the result tables and figures inline and saves a consolidated PKL bundle to Google Drive.**


In [ ]:
!pip -q install -r https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/b115ac27bf953e25e900defd68c03e608a15b06b/requirements-cemat-v4.txt


In [ ]:
import os
os.environ['CEMAT_REPEATS'] = '5'
os.environ['CEMAT_BOOTSTRAPS'] = '3000'
os.environ['CEMAT_FORCE_RESTART'] = '0'
print('Scientific configuration: 5 repeats, 5 outer folds, 3000 bootstraps')


In [ ]:
import hashlib
import urllib.request

LOADER_COMMIT = '1c79ed6718d49afeb8589a3eb03c7c7624bb1c30'
EXPECTED_LOADER_SHA256 = 'ddb3e666d6f1ac8fa19e372773ee786cc3e3085c66ac7f175e3f92411eb0b705'
LOADER_URL = ('https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/'
              f'{LOADER_COMMIT}/src/v4/verified_loader.py')
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
actual = hashlib.sha256(loader_bytes).hexdigest()
print('Loader commit:', LOADER_COMMIT)
print('Loader SHA256:', actual)
if actual != EXPECTED_LOADER_SHA256:
    raise RuntimeError(f'Loader checksum mismatch: {actual}')
exec(compile(loader_bytes.decode('utf-8'), LOADER_URL, 'exec'), globals(), globals())


In [ ]:
# Display final outputs inline and save a consolidated PKL bundle.
import json, os, pickle, shutil
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

RUNS_ROOT = Path('/content/drive/MyDrive/MAT-Appendix/cemat_v4_runs')
required = [
    'summary_mean_std.csv', 'consensus_metrics.csv', 'paired_bootstrap.csv',
    'final_decision.json', 'publication_audit.json',
    'all_repeated_nested_predictions.csv'
]
runs = [
    p for p in RUNS_ROOT.glob('cemat_stack_v4_*')
    if p.is_dir() and all((p / name).exists() for name in required)
]
if not runs:
    raise RuntimeError('No completed CEMAT V4 run found. Check the training cell above.')
RUN_DIR = max(runs, key=lambda p: p.stat().st_mtime)
print('Completed run:', RUN_DIR)

table_names = [
    'summary_mean_std.csv', 'consensus_metrics.csv', 'repeat_metrics.csv',
    'paired_bootstrap.csv', 'cohort_summary.csv', 'leakage_audit.csv',
    'fallback_audit.csv', 'all_repeated_nested_predictions.csv',
    'per_patient_consensus.csv'
]
tables = {}
for name in table_names:
    path = RUN_DIR / name
    if path.exists():
        tables[name] = pd.read_csv(path)

display(Markdown('## Primary repeat-level mean ± SD'))
display(tables['summary_mean_std.csv'])
display(Markdown('## Consensus metrics'))
display(tables['consensus_metrics.csv'])
display(Markdown('## Paired bootstrap comparisons'))
display(tables['paired_bootstrap.csv'])

json_names = ['final_decision.json', 'publication_audit.json', 'config.json']
json_data = {}
for name in json_names:
    path = RUN_DIR / name
    if path.exists():
        json_data[name] = json.loads(path.read_text(encoding='utf-8'))
        display(Markdown('## ' + name.replace('_', ' ').replace('.json', '').title()))
        print(json.dumps(json_data[name], indent=2))

figure_names = [
    'balanced_metric_comparison.png',
    'high_sensitivity_metric_comparison.png',
    'consensus_roc.png', 'consensus_pr.png', 'consensus_calibration.png'
]
figures = {}
for name in figure_names:
    path = RUN_DIR / name
    if path.exists():
        figures[name] = str(path)
        display(Markdown('## ' + name.replace('_', ' ').replace('.png', '').title()))
        display(Image(filename=str(path)))

fold_logs = {}
for path in sorted(RUN_DIR.glob('fold_log_repeat*_fold*.json')):
    fold_logs[path.name] = json.loads(path.read_text(encoding='utf-8'))

bundle = {
    'schema': 'cemat-stack-v4-complete-results-v1',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'run_directory': str(RUN_DIR),
    'source_commit': os.environ.get('CEMAT_SOURCE_COMMIT'),
    'source_sha256': os.environ.get('CEMAT_SOURCE_SHA256'),
    'loader_commit': LOADER_COMMIT,
    'loader_sha256': actual,
    'tables': tables,
    'json': json_data,
    'fold_logs': fold_logs,
    'figures': figures,
    'primary_evidence': 'summary_mean_std.csv repeat-level mean ± SD'
}
pkl_path = RUN_DIR / 'cemat_stack_v4_complete_results.pkl'
with pkl_path.open('wb') as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

latest_pkl = RUNS_ROOT / 'LATEST_CEMAT_V4_RESULTS.pkl'
shutil.copy2(pkl_path, latest_pkl)

display(Markdown('## PKL export complete'))
print('Run PKL:', pkl_path)
print('Latest PKL:', latest_pkl)
print('PKL size:', latest_pkl.stat().st_size, 'bytes')


## Saved PKL

- Run-specific: `MyDrive/MAT-Appendix/cemat_v4_runs/cemat_stack_v4_<hash>/cemat_stack_v4_complete_results.pkl`
- Latest copy: `MyDrive/MAT-Appendix/cemat_v4_runs/LATEST_CEMAT_V4_RESULTS.pkl`
